# SentinelAI: Hallucination Detector (final notebook)

Binary detector: **0 = clean response, 1 = hallucinated response** for three tasks (Data2txt, QA, Summary).

## What changed compared with the previous version
1. **Main fix: the model now sees the real source text.** Before, ModernBERT's input was `task + dataset name (e.g. "CNN/DM")` and the response, so it never saw the document it had to check against. Now the input is `Task + source_info` paired with the `response`. `source_info` is merged in by `source_id` and checked for gaps.
2. **The old response-only model is kept as a named comparison model** (same input as before), so the effect of the fix is measured on validation.
3. **Two simple models added:** a majority-class baseline and TF-IDF + Logistic Regression.
4. **Every model gets its own threshold, chosen on validation.** The final model is chosen on validation only.
5. **The test set is not touched until the final section**, and the model and threshold are saved to `selection.json` before it is used.
6. **Sensitivity presets fixed.** The old function silently ignored the mode names and returned a dict that broke pandas. The new presets are derived from validation and unknown mode names raise an error.
7. Removed wasteful or duplicate cells (`get_dummies` on the full text columns, repeated test runs).

**Runtime warning:** this notebook needs a **GPU** (it stops with a clear message if there is none, because CPU training would take 30+ hours). The source-aware model reads about 3x more tokens than the response-only model, so expect it to train much longer than the old ~24 minutes on a T4 (my estimate: 1 to 2 hours; set `PROFILE = "lite"` to roughly halve that).

**Crash safety:** it runs on Kaggle or Colab. Checkpoints are saved after every epoch and training resumes automatically after a crash, a disconnect or an "Aw, Snap" tab error. Finished models are skipped on re-run. Progress bars are switched off so the page stays light. The data files are checked for damage (cut-off uploads) before anything runs.

## 1. Setup and configuration

In [ ]:
# !pip install -q -U transformers datasets accelerate scikit-learn
# Optional: keep files after the Colab session ends
# from google.colab import drive; drive.mount('/content/drive')

In [ ]:
import os
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")   # use one GPU: keeps batch size and results predictable
import gc, json, ast, glob, shutil, random, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import transformers, datasets as hf_datasets
from IPython.display import display
from scipy.special import softmax
from sklearn.model_selection import GroupShuffleSplit
from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (precision_recall_fscore_support, accuracy_score, average_precision_score,
                             roc_auc_score, classification_report, confusion_matrix, ConfusionMatrixDisplay)
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding, set_seed)

warnings.filterwarnings("ignore", category=FutureWarning)
try:
    hf_datasets.disable_progress_bars()      # fewer outputs = lighter browser tab
except Exception:
    pass

# ---------------- configuration ----------------
SEED = 42
MODEL_NAME = "answerdotai/ModernBERT-base"

# ---- where are we running? Models are saved somewhere that survives a tab crash ----
if os.path.isdir("/kaggle/working"):                     # Kaggle
    BASE = "/kaggle/working"
elif os.path.isdir("/content/drive/MyDrive"):            # Colab with Google Drive mounted (best choice)
    BASE = "/content/drive/MyDrive/sentinel_ai"
else:                                                    # plain Colab (files vanish when the session ends)
    BASE = "/content"
os.makedirs(BASE, exist_ok=True)

def find_file(name):
    candidates = [f"/content/{name}", f"/content/drive/MyDrive/{name}",
                  f"/content/drive/MyDrive/sentinel_ai/{name}", f"{BASE}/{name}"]
    candidates += glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    for c in candidates:
        if os.path.exists(c):
            return c
    raise FileNotFoundError(f"Could not find {name}. Looked in: {candidates}")

RESPONSE_PATH = find_file("response.jsonl")
SOURCE_PATH = find_file("source_info.jsonl")
WORK_DIR = os.path.join(BASE, "sentinel_work")       # trained models and checkpoints
FINAL_DIR = os.path.join(BASE, "SentinelAI_final")   # final deliverable folder
print("Running from:", BASE, "\nData:", RESPONSE_PATH, "|", SOURCE_PATH)

PROFILE = "full"      # "full" = 2048 tokens, 3 epochs | "lite" = 1024 tokens, 2 epochs (about half the time)
MAX_LEN, EPOCHS = (2048, 3) if PROFILE == "full" else (1024, 2)   # source text is cut first if a pair is longer
TRAIN_BS = 2          # per-device batch; long inputs need a small batch on a T4
GRAD_ACCUM = 8        # effective batch size = 2 x 8 = 16
EVAL_BS = 4
LR = 2e-5
WEIGHT_DECAY = 0.01
WARMUP = 0.1
USE_FP16 = True       # set False if training loss becomes NaN
EVAL_FP16 = False     # full precision at prediction time avoids NaN logits
RETRAIN = False       # False = reuse finished models and resume interrupted training; True = start from scratch
ALLOW_CPU = False     # leave False: CPU training would take 30+ hours

os.makedirs(WORK_DIR, exist_ok=True)
set_seed(SEED)
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise RuntimeError("No GPU found. Colab: Runtime > Change runtime type > T4 GPU. "
                       "Kaggle: Settings > Accelerator > GPU. CPU training would take 30+ hours.")
print("torch", torch.__version__, "| transformers", transformers.__version__, "| datasets", hf_datasets.__version__)
print("GPU available:", torch.cuda.is_available(), "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no GPU")
print(f"PROFILE={PROFILE} | MAX_LEN={MAX_LEN} | EPOCHS={EPOCHS}")

## 2. Load data and merge `source_info`
The merge is `many_to_one` on `source_id`, and we stop if any response ends up without its source text.

In [ ]:
def check_jsonl(path):
    """Stop early if the file is damaged (for example a cut-off upload)."""
    good = bad = 0
    first_bad = None
    with open(path, encoding="utf-8", errors="replace") as f:
        for n, line in enumerate(f, 1):
            if not line.strip():
                continue
            try:
                json.loads(line); good += 1
            except Exception:
                bad += 1
                first_bad = first_bad or n
    if bad:
        raise ValueError(f"{path} is damaged: {bad} unreadable line(s), first at line {first_bad} "
                         f"(valid lines: {good}). It is probably cut off by an interrupted upload. "
                         "Re-upload the original file and run this cell again.")
    return good

print("response.jsonl valid lines:", check_jsonl(RESPONSE_PATH), "| source_info.jsonl valid lines:", check_jsonl(SOURCE_PATH))
responses = pd.read_json(RESPONSE_PATH, lines=True)
source_infos = pd.read_json(SOURCE_PATH, lines=True)
print("responses:", responses.shape, "| source_infos:", source_infos.shape)

df = pd.merge(responses, source_infos, on="source_id", how="left", validate="many_to_one")

# ---- the key check for the fix: every response must have its source text ----
assert len(df) == len(responses), "Merge changed the number of rows"
assert df["source_info"].notna().all(), "Some responses have no source_info after the merge"
print("merged:", df.shape, "| missing source_info:", int(df["source_info"].isna().sum()))

## 3. Labels, source text cleaning

In [ ]:
def parse_labels(x):
    if isinstance(x, str):
        x = ast.literal_eval(x)
    return x if isinstance(x, list) else []

df["labels"] = df["labels"].apply(parse_labels)
df["is_hallucinated"] = df["labels"].apply(lambda x: int(len(x) > 0))

def source_to_text(x):
    """Turn source_info (text, list or dict) into one plain string."""
    if isinstance(x, (list, dict)):
        val = x
    else:
        try:
            val = ast.literal_eval(x)
        except Exception:
            return str(x)
    if isinstance(val, list):
        return " ".join(map(str, val))
    if isinstance(val, dict):
        return json.dumps(val, ensure_ascii=False)
    return str(val)

df["source_info"] = df["source_info"].apply(source_to_text)
df["response"] = df["response"].astype(str)

assert (df["source_info"].str.strip().str.len() > 0).all(), "Empty source_info found"
print(df["source_info"].map(type).value_counts().to_string())

summary = (df.groupby("model")["is_hallucinated"].agg(["count", "sum", "mean"])
             .rename(columns={"count": "responses", "sum": "hallucinated", "mean": "error_rate"}))
summary["error_rate"] = (summary["error_rate"] * 100).round(2).astype(str) + "%"
display(summary)
print("Overall hallucination rate: %.2f%%" % (df["is_hallucinated"].mean() * 100))

## 4. Train / validation / test split
* `test` is the official test split and stays untouched until section 12.
* Validation is 20% of the official train split, grouped by `source_id` so the same source never appears in both train and validation.

In [ ]:
train_full = df[df["split"] == "train"].reset_index(drop=True)
df_test = df[df["split"] == "test"].reset_index(drop=True)

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx, va_idx = next(gss.split(train_full, groups=train_full["source_id"]))
df_tr = train_full.iloc[tr_idx].reset_index(drop=True)
df_val = train_full.iloc[va_idx].reset_index(drop=True)

assert len(set(df_tr["source_id"]) & set(df_val["source_id"])) == 0, "train/val share sources"
print("train/test shared source_ids:", len(set(train_full["source_id"]) & set(df_test["source_id"])))

for name, d in [("train", df_tr), ("validation", df_val), ("test", df_test)]:
    print(f"{name:11} rows={len(d):6} | hallucinated={d['is_hallucinated'].mean()*100:5.1f}% | "
          + ", ".join(f"{t}={n}" for t, n in d["task_type"].value_counts().items()))

## 5. Model inputs (this is where the fix lives)

| Mode | First segment | Second segment |
|---|---|---|
| `response_only` (old, kept for comparison) | `"<task> to <dataset name>"` | response |
| `source_aware` (new) | `"Task: <task>. Source: <source_info>"` | response |

If a pair is longer than `MAX_LEN`, the **source** is cut from the end and the response stays whole (when it is the shorter part). The task prefix is at the start, so it is never cut.

In [ ]:
def build_inputs(d, mode):
    if mode == "response_only":
        first = (d["task_type"].astype(str) + " to " + d["source"].astype(str)).tolist()
    elif mode == "source_aware":
        first = ("Task: " + d["task_type"].astype(str) + ". Source: " + d["source_info"].astype(str)).tolist()
    else:
        raise ValueError(f"Unknown mode: {mode}")
    return first, d["response"].astype(str).tolist()

def make_dataset(d, mode, tokenizer, max_len=MAX_LEN):
    first, second = build_inputs(d, mode)
    ds = Dataset.from_dict({"first": first, "second": second,
                            "labels": d["is_hallucinated"].astype(int).tolist()})
    def tok(batch):
        return tokenizer(batch["first"], batch["second"], truncation="longest_first", max_length=max_len)
    return ds.map(tok, batched=True, remove_columns=["first", "second"])

# ---- how long are the inputs? (lengths only, no labels used) ----
len_tok = AutoTokenizer.from_pretrained(MODEL_NAME)
lengths = []
for mode in ["response_only", "source_aware"]:
    f, s = build_inputs(df, mode)
    n = []
    for i in range(0, len(f), 500):
        enc = len_tok(f[i:i+500], s[i:i+500], add_special_tokens=True)["input_ids"]
        n.extend(len(x) for x in enc)
    lengths.append(pd.Series(n, name=mode))
len_df = pd.concat(lengths, axis=1)
len_df["task_type"] = df["task_type"].values

stats = len_df.groupby("task_type")["source_aware"].agg(
    median="median", p95=lambda x: np.percentile(x, 95), max="max",
    pct_over_limit=lambda x: (x > MAX_LEN).mean() * 100).round(1)
print("Source-aware input length in tokens (MAX_LEN = %d):" % MAX_LEN)
display(stats)
print("Mean tokens | response_only: %.0f | source_aware: %.0f" % (len_df["response_only"].mean(), len_df["source_aware"].mean()))

## 6. Helper functions (training, prediction, thresholds, metrics)

In [ ]:
def cleanup_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    logits = np.nan_to_num(logits, nan=0.0, posinf=11.0, neginf=-11.0)
    probs = softmax(logits, axis=1)[:, 1]
    preds = (probs >= 0.5).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", pos_label=1, zero_division=0)
    return {"accuracy": accuracy_score(labels, preds), "precision": p, "recall": r, "f1": f1,
            "pr_auc": average_precision_score(labels, probs)}

def train_model(run_name, mode):
    """Fine-tune ModernBERT on df_tr, pick the best epoch on validation, save to disk, return the folder."""
    run_dir = os.path.join(WORK_DIR, run_name)
    final_path = os.path.join(run_dir, "final")
    if RETRAIN and os.path.isdir(run_dir):
        shutil.rmtree(run_dir)
    if (not RETRAIN) and os.path.exists(os.path.join(final_path, "config.json")):
        print(f"[{run_name}] found a finished model, skipping training: {final_path}")
        return final_path

    set_seed(SEED)
    tok = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds = make_dataset(df_tr, mode, tok)
    val_ds = make_dataset(df_val, mode, tok)
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    kwargs = dict(
        output_dir=os.path.join(run_dir, "checkpoints"),
        learning_rate=LR, weight_decay=WEIGHT_DECAY, num_train_epochs=EPOCHS,
        per_device_train_batch_size=TRAIN_BS, per_device_eval_batch_size=EVAL_BS,
        gradient_accumulation_steps=GRAD_ACCUM,
        fp16=bool(USE_FP16 and torch.cuda.is_available()),
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=2,
        load_best_model_at_end=True, metric_for_best_model="pr_auc", greater_is_better=True,
        logging_steps=100, disable_tqdm=True, report_to="none", seed=SEED)
    try:
        args = TrainingArguments(**kwargs, warmup_ratio=WARMUP)
    except TypeError:                       # newer versions may rename this option
        args = TrainingArguments(**kwargs)

    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      processing_class=tok, data_collator=DataCollatorWithPadding(tok),
                      compute_metrics=compute_metrics)
    ckpt_root = os.path.join(run_dir, "checkpoints")
    can_resume = os.path.isdir(ckpt_root) and any(d.startswith("checkpoint-") for d in os.listdir(ckpt_root))
    print(f"[{run_name}] training ({mode}) | " + ("RESUMING from the last saved epoch" if can_resume else "starting fresh"))
    trainer.train(resume_from_checkpoint=True if can_resume else None)

    trainer.save_model(final_path)
    tok.save_pretrained(final_path)
    with open(os.path.join(run_dir, "training_log.json"), "w") as f:
        json.dump(trainer.state.log_history, f, indent=1, default=str)
    del trainer, model
    cleanup_gpu()
    return final_path

def predict_probs(model_dir, mode, d):
    """Probability of 'hallucinated' for every row of d, using a saved model."""
    tok = AutoTokenizer.from_pretrained(model_dir)
    model = AutoModelForSequenceClassification.from_pretrained(model_dir)
    ds = make_dataset(d, mode, tok)
    args = TrainingArguments(output_dir=os.path.join(WORK_DIR, "tmp_eval"),
                             per_device_eval_batch_size=EVAL_BS,
                             fp16=bool(EVAL_FP16 and torch.cuda.is_available()), report_to="none")
    trainer = Trainer(model=model, args=args, processing_class=tok, data_collator=DataCollatorWithPadding(tok))
    logits = trainer.predict(ds).predictions
    bad = int((~np.isfinite(logits)).sum())
    if bad:
        print(f"WARNING: {bad} non-finite logits were replaced with 0")
    logits = np.nan_to_num(logits, nan=0.0, posinf=11.0, neginf=-11.0)
    del trainer, model
    cleanup_gpu()
    return softmax(logits, axis=1)[:, 1]

THRESHOLDS = np.round(np.arange(0.05, 0.951, 0.01), 2)

def metrics_at(y, probs, thr):
    pred = (probs >= thr).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", pos_label=1, zero_division=0)
    return {"threshold": float(thr), "accuracy": float(accuracy_score(y, pred)), "precision": float(p),
            "recall": float(r), "f1": float(f),
            "pr_auc": float(average_precision_score(y, probs)), "roc_auc": float(roc_auc_score(y, probs))}

def sweep_thresholds(y, probs):
    rows = []
    for t in THRESHOLDS:
        pred = (probs >= t).astype(int)
        p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", pos_label=1, zero_division=0)
        rows.append({"threshold": float(t), "accuracy": accuracy_score(y, pred), "precision": p, "recall": r, "f1": f})
    return pd.DataFrame(rows)

def best_threshold(sweep):
    return float(sweep.loc[sweep["f1"].idxmax(), "threshold"])

def per_task_metrics(d, probs, thr):
    pred = (probs >= thr).astype(int)
    out = {}
    for t in sorted(d["task_type"].unique()):
        m = (d["task_type"] == t).values
        p, r, f, _ = precision_recall_fscore_support(d["is_hallucinated"].values[m], pred[m],
                                                     average="binary", pos_label=1, zero_division=0)
        out[t] = {"precision": float(p), "recall": float(r), "f1": float(f), "n": int(m.sum())}
    return out

## 7. Simple comparison models (trained on train only, evaluated on validation)
* **Majority baseline:** always predicts "clean". It is the floor every real model must beat.
* **TF-IDF + Logistic Regression:** a classic text model on `task + response`.

In [ ]:
NAME_MAJ = "Majority baseline (always Clean)"
NAME_TFIDF = "TF-IDF + Logistic Regression"
NAME_RESP = "ModernBERT response-only (old input)"
NAME_SRC = "ModernBERT source-aware (new input)"

y_val = df_val["is_hallucinated"].values
y_test = df_test["is_hallucinated"].values

val_probs, sweeps, val_thr = {}, {}, {}

# ---- Model 1: majority baseline ----
val_probs[NAME_MAJ] = np.zeros(len(df_val))
val_thr[NAME_MAJ] = 0.5

# ---- Model 2: TF-IDF + Logistic Regression ----
def tfidf_text(d):
    return (d["task_type"].astype(str) + " " + d["response"].astype(str)).tolist()

tfidf = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)
X_tr = tfidf.fit_transform(tfidf_text(df_tr))
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=SEED)
lr.fit(X_tr, df_tr["is_hallucinated"].values)

val_probs[NAME_TFIDF] = lr.predict_proba(tfidf.transform(tfidf_text(df_val)))[:, 1]
sweeps[NAME_TFIDF] = sweep_thresholds(y_val, val_probs[NAME_TFIDF])
val_thr[NAME_TFIDF] = best_threshold(sweeps[NAME_TFIDF])
print("TF-IDF + LogReg ready | best validation threshold:", val_thr[NAME_TFIDF])

## 8. Train ModernBERT twice (the comparison that matters)
Both runs use the same settings, split, seed and training function. Only the input changes.

Training uses **train only**. Validation is used to pick the best epoch (by PR-AUC, which does not depend on a threshold). Models are saved to `WORK_DIR`.

In [ ]:
dir_resp = train_model("response_only", "response_only")   # old input (dataset name + response)
print("saved:", dir_resp)

In [ ]:
dir_src = train_model("source_aware", "source_aware")       # new input (source text + response)
print("saved:", dir_src)

## 9. Validation comparison of all four models
Each model is judged at **its own best threshold, chosen on validation**, so no model is handicapped by the default 0.5.

In [ ]:
model_dirs = {NAME_RESP: dir_resp, NAME_SRC: dir_src}
model_modes = {NAME_RESP: "response_only", NAME_SRC: "source_aware"}

for name in [NAME_RESP, NAME_SRC]:
    val_probs[name] = predict_probs(model_dirs[name], model_modes[name], df_val)
    sweeps[name] = sweep_thresholds(y_val, val_probs[name])
    val_thr[name] = best_threshold(sweeps[name])

ORDER = [NAME_MAJ, NAME_TFIDF, NAME_RESP, NAME_SRC]
val_metrics = {n: metrics_at(y_val, val_probs[n], val_thr[n]) for n in ORDER}
val_default = {n: metrics_at(y_val, val_probs[n], 0.5) for n in ORDER}
val_task = {n: per_task_metrics(df_val, val_probs[n], val_thr[n]) for n in ORDER}

val_table = pd.DataFrame(val_metrics).T[["threshold", "accuracy", "precision", "recall", "f1", "pr_auc"]]
val_table["f1_at_0.5"] = [val_default[n]["f1"] for n in ORDER]
for t in sorted(df_val["task_type"].unique()):
    val_table[f"f1_{t}"] = [val_task[n][t]["f1"] for n in ORDER]
val_table = val_table.round(4)
print("=== VALIDATION: all models (each at its own validation-chosen threshold) ===")
display(val_table)

# ---- chart ----
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
short = ["Majority", "TF-IDF+LR", "ModernBERT\nresponse-only", "ModernBERT\nsource-aware"]
axes[0].bar(short, val_table["f1"].values, color=["#bbbbbb", "#8fb8de", "#4c78a8", "#e45756"])
axes[0].set_title("Validation F1 (overall)"); axes[0].set_ylim(0, 1)
for i, v in enumerate(val_table["f1"].values):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha="center")
tasks = sorted(df_val["task_type"].unique())
w = 0.2
for i, n in enumerate(ORDER):
    axes[1].bar(np.arange(len(tasks)) + i * w, [val_task[n][t]["f1"] for t in tasks], w, label=short[i].replace("\n", " "))
axes[1].set_xticks(np.arange(len(tasks)) + 1.5 * w); axes[1].set_xticklabels(tasks)
axes[1].set_title("Validation F1 per task"); axes[1].set_ylim(0, 1); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## 10. Select the final model and threshold (validation only)
Rule: of the two ModernBERT variants, keep the one with the higher **validation F1** at its own validation-chosen threshold. The decision is written to `selection.json` **before** the test set is used.

In [ ]:
gain = val_metrics[NAME_SRC]["f1"] - val_metrics[NAME_RESP]["f1"]
print(f"Validation F1  response-only: {val_metrics[NAME_RESP]['f1']:.4f} | source-aware: {val_metrics[NAME_SRC]['f1']:.4f} | gain: {gain:+.4f}")

final_name = max([NAME_RESP, NAME_SRC], key=lambda n: val_metrics[n]["f1"])
FINAL_MODE = model_modes[final_name]
FINAL_MODEL_DIR = model_dirs[final_name]
FINAL_THRESHOLD = val_thr[final_name]
print("\nSELECTED MODEL    :", final_name)
print("INPUT MODE        :", FINAL_MODE)
print("FINAL THRESHOLD   :", FINAL_THRESHOLD)
if final_name == NAME_RESP:
    print("Note: the source-aware model did NOT beat the response-only model on validation.")

selection = {"selected_model": final_name, "input_mode": FINAL_MODE, "threshold": FINAL_THRESHOLD,
             "selected_on": "validation F1 (threshold also chosen on validation)",
             "validation_metrics": val_metrics[final_name],
             "validation_f1_gain_source_aware_vs_response_only": float(gain)}
os.makedirs(FINAL_DIR, exist_ok=True)
with open(os.path.join(FINAL_DIR, "selection.json"), "w") as f:
    json.dump(selection, f, indent=2)
print("\nSelection locked and saved. The test set has not been used yet.")

### Sensitivity presets (derived from validation)
Presets give users a simple knob. They are built from the selected model's **validation** curve:
* **Catch as many as possible:** the highest threshold that gives validation recall of at least 0.85, and at least 5 points more recall than the balanced setting
* **Balanced detection:** the final threshold (best validation F1)
* **Only flag stronger signals:** the lowest threshold that gives validation precision of at least 0.80, and at least 5 points more precision than the balanced setting
* If a target cannot be reached, the preset falls back to the lowest or highest threshold tested.

Unknown mode names raise an error instead of silently using a default.

In [ ]:
def build_presets(sweep, final_thr, base_recall=0.85, base_precision=0.80):
    at = sweep.iloc[(sweep["threshold"] - final_thr).abs().argmin()]
    # targets are always a bit stricter than what the balanced threshold already gives
    target_recall = max(base_recall, min(0.97, float(at["recall"]) + 0.05))
    target_precision = max(base_precision, min(0.97, float(at["precision"]) + 0.05))
    hi_r = sweep[sweep["recall"] >= target_recall]
    t_catch = float(hi_r["threshold"].max()) if len(hi_r) else float(sweep["threshold"].min())
    hi_p = sweep[sweep["precision"] >= target_precision]
    t_strict = float(hi_p["threshold"].min()) if len(hi_p) else float(sweep["threshold"].max())
    return {"Catch as many as possible": min(t_catch, final_thr),
            "Balanced detection": final_thr,
            "Only flag stronger signals": max(t_strict, final_thr)}

SENTINEL_PRESETS = build_presets(sweeps[final_name], FINAL_THRESHOLD)

def sentinel_predict(probs, sensitivity_mode="Balanced detection", presets=SENTINEL_PRESETS):
    if sensitivity_mode not in presets:
        raise ValueError(f"Unknown mode '{sensitivity_mode}'. Choose from: {list(presets)}")
    thr = presets[sensitivity_mode]
    return {"predictions": [int(p >= thr) for p in probs], "threshold": thr, "sensitivity": sensitivity_mode}

rows = []
for mode_name, thr in SENTINEL_PRESETS.items():
    m = metrics_at(y_val, val_probs[final_name], thr)
    rows.append({"mode": mode_name, **{k: round(m[k], 4) for k in ["threshold", "accuracy", "precision", "recall", "f1"]}})
print("=== SENSITIVITY PRESETS on VALIDATION ===")
display(pd.DataFrame(rows))

## 11. Final test evaluation (run once)
The selected model and its locked threshold are applied to the official test split. Nothing is tuned after this point.

In [ ]:
test_probs_final = predict_probs(FINAL_MODEL_DIR, FINAL_MODE, df_test)
test_pred_final = (test_probs_final >= FINAL_THRESHOLD).astype(int)

test_overall = metrics_at(y_test, test_probs_final, FINAL_THRESHOLD)
test_task = per_task_metrics(df_test, test_probs_final, FINAL_THRESHOLD)

print(f"=== FINAL TEST RESULTS | {final_name} | threshold = {FINAL_THRESHOLD} ===")
print("Accuracy: %.4f | Precision: %.4f | Recall: %.4f | F1: %.4f | PR-AUC: %.4f"
      % (test_overall["accuracy"], test_overall["precision"], test_overall["recall"], test_overall["f1"], test_overall["pr_auc"]))

print("\nClassification report:")
print(classification_report(y_test, test_pred_final, target_names=["Clean", "Hallucinated"], zero_division=0))
report_dict = classification_report(y_test, test_pred_final, target_names=["Clean", "Hallucinated"],
                                    output_dict=True, zero_division=0)

cm = confusion_matrix(y_test, test_pred_final)
fig, ax = plt.subplots(figsize=(5, 4.5))
ConfusionMatrixDisplay(cm, display_labels=["Clean", "Hallucinated"]).plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title(f"Test confusion matrix (threshold={FINAL_THRESHOLD})")
fig.savefig(os.path.join(FINAL_DIR, "confusion_matrix.png"), dpi=150, bbox_inches="tight")
plt.show()
print("Rows = true label, columns = predicted label. False alarms:", int(cm[0, 1]), "| Missed hallucinations:", int(cm[1, 0]))

print("\nPer task:")
display(pd.DataFrame(test_task).T.round(4))

### Test results for all four models (reporting only)
These numbers are for the report. They were **not** used to choose anything, because the model and every threshold were locked in section 10. Each model uses its own validation-chosen threshold.

In [ ]:
test_probs_all = {NAME_MAJ: np.zeros(len(df_test)),
                  NAME_TFIDF: lr.predict_proba(tfidf.transform(tfidf_text(df_test)))[:, 1],
                  final_name: test_probs_final}
other = NAME_RESP if final_name == NAME_SRC else NAME_SRC
test_probs_all[other] = predict_probs(model_dirs[other], model_modes[other], df_test)

test_metrics = {n: metrics_at(y_test, test_probs_all[n], val_thr[n]) for n in ORDER}
test_task_all = {n: per_task_metrics(df_test, test_probs_all[n], val_thr[n]) for n in ORDER}
test_table = pd.DataFrame(test_metrics).T[["threshold", "accuracy", "precision", "recall", "f1", "pr_auc"]]
for t in sorted(df_test["task_type"].unique()):
    test_table[f"f1_{t}"] = [test_task_all[n][t]["f1"] for n in ORDER]
test_table = test_table.round(4)

print("=== TEST: all models (thresholds locked from validation) ===")
display(test_table)

final_comparison = pd.concat({"validation": val_table[test_table.columns], "test": test_table}, names=["set", "model"])
os.makedirs(FINAL_DIR, exist_ok=True)
final_comparison.to_csv(os.path.join(FINAL_DIR, "model_comparison.csv"))

## 12. Save the final model, tokenizer, configuration and evaluation files

In [ ]:
# 1. model + tokenizer + model config
for fn in os.listdir(FINAL_MODEL_DIR):
    src = os.path.join(FINAL_MODEL_DIR, fn)
    if os.path.isfile(src):
        shutil.copy2(src, os.path.join(FINAL_DIR, fn))

# 2. SentinelAI configuration
sentinel_config = {
    "base_model": MODEL_NAME,
    "selected_model": final_name,
    "input_mode": FINAL_MODE,
    "input_format": ("first segment: 'Task: <task_type>. Source: <source_info>'; second segment: response"
                     if FINAL_MODE == "source_aware" else
                     "first segment: '<task_type> to <dataset name>'; second segment: response"),
    "label_map": {"0": "clean", "1": "hallucinated"},
    "final_threshold": FINAL_THRESHOLD,
    "sensitivity_presets": SENTINEL_PRESETS,
    "threshold_selected_on": "validation set (F1)",
    "max_length": MAX_LEN,
    "truncation": "longest_first (the source text is shortened first)",
    "training": {"learning_rate": LR, "epochs": EPOCHS, "effective_batch_size": TRAIN_BS * GRAD_ACCUM,
                 "weight_decay": WEIGHT_DECAY, "warmup_ratio": WARMUP, "fp16": USE_FP16, "seed": SEED,
                 "best_epoch_metric": "pr_auc on validation"},
    "split": "official train/test; validation = 20% of train grouped by source_id (GroupShuffleSplit, seed 42)",
    "versions": {"torch": torch.__version__, "transformers": transformers.__version__},
}
with open(os.path.join(FINAL_DIR, "sentinel_config.json"), "w") as f:
    json.dump(sentinel_config, f, indent=2)

# 3. evaluation results
evaluation = {
    "selected_model": final_name,
    "test_overall": test_overall,
    "test_per_task": test_task,
    "confusion_matrix": {"labels": ["Clean", "Hallucinated"], "matrix": cm.tolist()},
    "classification_report": report_dict,
    "validation_comparison": json.loads(val_table.to_json(orient="index")),
    "test_comparison_reporting_only": json.loads(test_table.to_json(orient="index")),
    "sensitivity_presets_validation": rows,
    "notes": [
        "Model and threshold were selected on validation only and written to selection.json before the test set was used.",
        "The test set was evaluated once for the selected model; other models' test scores are reporting only.",
        f"Hallucination rate differs by split: train {df_tr['is_hallucinated'].mean()*100:.1f}%, "
        f"validation {df_val['is_hallucinated'].mean()*100:.1f}%, test {df_test['is_hallucinated'].mean()*100:.1f}%. "
        "A threshold tuned on a higher-prevalence validation set tends to lower precision on a lower-prevalence test set.",
    ],
}
with open(os.path.join(FINAL_DIR, "evaluation_results.json"), "w") as f:
    json.dump(evaluation, f, indent=2)

# 4. validation threshold curve for the selected model
sweeps[final_name].round(4).to_csv(os.path.join(FINAL_DIR, "validation_threshold_curve.csv"), index=False)

print("Saved:", sorted(os.listdir(FINAL_DIR)))

## 13. Reload check and inference helper
Loads the **saved** model from `FINAL_DIR` and confirms it reproduces the validation probabilities. This proves the saved files work. The same class can be used for new inputs.

In [ ]:
class SentinelDetector:
    """Loads the saved SentinelAI model and scores (task, source, response) triples."""
    def __init__(self, model_dir=FINAL_DIR):
        self.cfg = json.load(open(os.path.join(model_dir, "sentinel_config.json")))
        self.tok = AutoTokenizer.from_pretrained(model_dir)
        self.model = AutoModelForSequenceClassification.from_pretrained(model_dir)
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.model.to(self.device).eval()
        self.max_len = self.cfg["max_length"]

    def _first(self, task_type, source_info, dataset_name):
        if self.cfg["input_mode"] == "source_aware":
            return f"Task: {task_type}. Source: {source_to_text(source_info)}"
        return f"{task_type} to {dataset_name}"

    @torch.no_grad()
    def predict_proba(self, task_types, source_infos, responses, dataset_names=None, batch_size=4):
        dataset_names = dataset_names or [""] * len(responses)
        firsts = [self._first(t, s, d) for t, s, d in zip(task_types, source_infos, dataset_names)]
        probs = []
        for i in range(0, len(responses), batch_size):
            enc = self.tok(firsts[i:i+batch_size], [str(r) for r in responses[i:i+batch_size]],
                           truncation="longest_first", max_length=self.max_len,
                           padding=True, return_tensors="pt").to(self.device)
            logits = self.model(**enc).logits.float().cpu().numpy()
            probs.extend(softmax(np.nan_to_num(logits), axis=1)[:, 1].tolist())
        return np.array(probs)

    def predict(self, task_types, source_infos, responses, dataset_names=None, mode="Balanced detection"):
        probs = self.predict_proba(task_types, source_infos, responses, dataset_names)
        out = sentinel_predict(probs, mode, presets=self.cfg["sensitivity_presets"])
        out["probabilities"] = probs.tolist()
        return out

detector = SentinelDetector(FINAL_DIR)

check = df_val.sample(40, random_state=SEED)                      # validation rows, not test
idx = check.index.values
reloaded = detector.predict_proba(check["task_type"].tolist(), check["source_info"].tolist(),
                                  check["response"].tolist(), check["source"].tolist())
original = val_probs[final_name][idx]
max_diff = float(np.abs(reloaded - original).max())
print("Largest difference between saved-model and original probabilities: %.5f" % max_diff)
print("Reload check:", "PASSED" if max_diff < 0.02 else "CHECK THIS: differences are larger than expected")

## 14. Zip the deliverable

In [ ]:
archive = shutil.make_archive(os.path.join(BASE, "SentinelAI_final"), "zip", FINAL_DIR)
print("Created:", archive)
print("Contents:", sorted(os.listdir(FINAL_DIR)))
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    pass